# k = 1 Computing Weingarten Coefficients

In [ ]:
# UNCOMMENT FOR INSTALLATION OF PACKAGES
# %pip install sympy
# %pip install haarpy
# %pip install numpy

In [2]:
import haarpy as hp
import sympy as sp
import itertools

from sympy.utilities.iterables import partitions
from sympy.combinatorics import Permutation
from sympy import latex

In [3]:
# initialize parameters and helpful datastructures
d = sp.Symbol('d')
k = 4

In [4]:
# Compute the Weingarten coefficients for all cycle types of S_k
wg_coeffs = {} # track the coefficients
cycle_keys = {} # label the cycle equiv classes

for i, p in enumerate(partitions(k)):
	# obtain the corresponding cyclic group
	cycle_type = []
	for part_size, count in sorted(p.items(), reverse=True):
		cycle_type.extend([part_size] * count)
	cycle_type = tuple(cycle_type)

	# weingarten function for each cycle type
	wg = hp.weingarten_unitary(cycle_type, d)
	wg = sp.factor(sp.simplify(wg)) # mainly for my sanity
	wg_coeffs[cycle_type] = wg

	cycle_keys[cycle_type] = i

In [5]:
# view results
for cycle_type, coeff in wg_coeffs.items():
	print(f"Cycle-type {str(cycle_type):<15} : {coeff}")

Cycle-type (4,)            : -5/(d*(d - 3)*(d - 2)*(d - 1)*(d + 1)*(d + 2)*(d + 3))
Cycle-type (3, 1)          : (2*d**2 - 3)/(d**2*(d - 3)*(d - 2)*(d - 1)*(d + 1)*(d + 2)*(d + 3))
Cycle-type (2, 2)          : (d**2 + 6)/(d**2*(d - 3)*(d - 2)*(d - 1)*(d + 1)*(d + 2)*(d + 3))
Cycle-type (2, 1, 1)       : -1/(d*(d - 3)*(d - 1)*(d + 1)*(d + 3))
Cycle-type (1, 1, 1, 1)    : (d**4 - 8*d**2 + 6)/(d**2*(d - 3)*(d - 2)*(d - 1)*(d + 1)*(d + 2)*(d + 3))


In [ ]:
# For simplicity, I have also indexed each cycle equivalence class
print(cycle_keys)

{(4,): 0, (3, 1): 1, (2, 2): 2, (2, 1, 1): 3, (1, 1, 1, 1): 4}


In [ ]:
# Method to compute the cycle type of pi^-1 * sigma in S_k as a sorted descending tuple
def get_cycle_type(permutation:Permutation, k: int) -> tuple:
	# get cycle structure of the product permutation
	struct = permutation.cycle_structure

	# account for fixed points
	total_elements_in_cycles = sum(length * count for length, count in struct.items())
	fixed_count = k - total_elements_in_cycles

	# get list
	cycle_lengths = []
	for length, count in struct.items():
		cycle_lengths.extend([length] * count)
	if fixed_count > 0:
		cycle_lengths.extend([1] * fixed_count)

	# format
	return tuple(sorted(cycle_lengths, reverse=True))

In [ ]:
# Compute the coefficients c_{pi} for all pi in S_k
# stores the coefficients as a dictionary with keys as permutations and values as the coefficients
c = {}

for pi in itertools.permutations(range(k)):
	pi_perm = Permutation(pi)
	c_pi = sp.core.mul.Mul() - 1 # initialize sum; default value is 1, so adjust the value by -1

	for sigma in itertools.permutations(range(k)):
		sigma_perm = Permutation(sigma)
		prod = pi_perm**-1 * sigma_perm
		cycle_type = get_cycle_type(prod, k)

		try:
			c_pi_sigma = wg_coeffs[cycle_type]
		except KeyError:
			c_pi_sigma = 0  # if cycle type isn't in the coefficients, I will assume it's 0

		# obtain the Tr(V_sigma^{\dagger} O)
		# since Tr (V_sigma^{\dagger} O) is not categorized by conjugacy class, the equation will become quite long here
		sigma_dagger = sigma_perm**-1
		tr = sp.Symbol(f'\\text{{Tr}}(V_{sigma_dagger}' + 'O)')

		c_pi += c_pi_sigma * tr

	c[pi] = sp.factor(sp.simplify(c_pi))

In [9]:
# View coefficients for all permutations
for pi in itertools.permutations(range(k)):
	print(f"c_{pi} = ${latex(c[pi])}$")

c_(0, 1, 2, 3) = $\frac{- 5 \text{Tr}(V_(0 1 2 3)O) d - 5 \text{Tr}(V_(0 1 3 2)O) d + 2 \text{Tr}(V_(0 1 3)O) d^{2} - 3 \text{Tr}(V_(0 1 3)O) + \text{Tr}(V_(0 1)(2 3)O) d^{2} + 6 \text{Tr}(V_(0 1)(2 3)O) - 5 \text{Tr}(V_(0 2 1 3)O) d - 5 \text{Tr}(V_(0 2 3 1)O) d + 2 \text{Tr}(V_(0 2 3)O) d^{2} - 3 \text{Tr}(V_(0 2 3)O) + \text{Tr}(V_(0 2)(1 3)O) d^{2} + 6 \text{Tr}(V_(0 2)(1 3)O) - 5 \text{Tr}(V_(0 3 1 2)O) d + 2 \text{Tr}(V_(0 3 1)O) d^{2} - 3 \text{Tr}(V_(0 3 1)O) - 5 \text{Tr}(V_(0 3 2 1)O) d + 2 \text{Tr}(V_(0 3 2)O) d^{2} - 3 \text{Tr}(V_(0 3 2)O) + \text{Tr}(V_(0 3)(1 2)O) d^{2} + 6 \text{Tr}(V_(0 3)(1 2)O) - \text{Tr}(V_(0 3)O) d^{3} + 4 \text{Tr}(V_(0 3)O) d + 2 \text{Tr}(V_(1 2 3)O) d^{2} - 3 \text{Tr}(V_(1 2 3)O) + 2 \text{Tr}(V_(1 3 2)O) d^{2} - 3 \text{Tr}(V_(1 3 2)O) - \text{Tr}(V_(1 3)O) d^{3} + 4 \text{Tr}(V_(1 3)O) d - \text{Tr}(V_(2 3)O) d^{3} + 4 \text{Tr}(V_(2 3)O) d + 2 \text{Tr}(V_(3)(0 1 2)O) d^{2} - 3 \text{Tr}(V_(3)(0 1 2)O) - \text{Tr}(V_(3)(0 1)O) d^{3} + 4 \